# 🚀 SpaceX Falcon 9 – Notebook 2: Web Scraping (Wikipedia)

Collect Falcon 9 historical launch records from Wikipedia using BeautifulSoup.

In [ ]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

print("Libraries ready!")

## Helper functions

In [ ]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg")+2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not(column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

## Scrape Wikipedia

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
response = requests.get(static_url)
soup = BeautifulSoup(response.text, 'html.parser')
print("Page fetched:", soup.title.string)

In [ ]:
# Extract all tables
html_tables = soup.find_all('table', {'class': 'wikitable plainrowheaders collapsible'})
print("Tables found:", len(html_tables))

# Get column names from first table
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print("Columns:", column_names)

In [ ]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Detail']
launch_dict['Version Booster'] = None
launch_dict['Booster landing'] = None

for key in launch_dict:
    launch_dict[key] = []

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            datatimelist      = date_time(row[0])
            Date              = datatimelist[0].strip(',')
            Time              = datatimelist[1]
            BoosterVersion    = booster_version(row[1])
            launch_site       = row[2].a.string if row[2].a else ''
            landing_status_v  = landing_status(row[-1])
            PayloadMass_v     = get_mass(row[5])
            try:
                orbit = row[6].a.string if row[6].a else row[6].string
            except:
                orbit = ''

            launch_dict['Flight No.'].append(flight_number)
            launch_dict['Launch site'].append(launch_site)
            launch_dict['Payload'].append(row[3].a.string if row[3].a else '')
            launch_dict['Payload mass'].append(PayloadMass_v)
            launch_dict['Orbit'].append(orbit)
            launch_dict['Customer'].append(row[7].a.string if row[7].a else '')
            launch_dict['Launch outcome'].append(list(row[8].strings)[0])
            launch_dict['Version Booster'].append(BoosterVersion)
            launch_dict['Booster landing'].append(landing_status_v)
            launch_dict['Date'].append(Date)
            launch_dict['Time'].append(Time)

df_wiki = pd.DataFrame(launch_dict)
print("Scraped rows:", len(df_wiki))
df_wiki.to_csv('spacex_web_scraped.csv', index=False)
print("Saved → spacex_web_scraped.csv")
df_wiki.head()